In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Task 1: Write text_to_bytes()

In [ ]:
def text_to_bytes(text):
    byte_array = bytearray(text, "utf-8")
    return list(byte_array)

In [ ]:
text = "the cat in the hat"

tokens = text_to_bytes(text)

print(tokens)

[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]


## Checkpoint 1

Answer the following questions:

1. What does UTF-8 do? **It converts text characters into a sequence of bytes**
2. What is the token ID for the character `t` in our example? **116**
3. Why are the initial IDs between 0 and 255? **Each byte can hold one of 256 possible values**
4. What does `text_to_bytes()` return? **A list of integers, where each integer corresponds to the byte value of the encoded characters in the input string**

# Task 2: Write get_frequent_pair

In [ ]:
from collections import Counter

def get_frequent_pair(tokens):
    if len(tokens) < 2:
        return None
    pairs = list(zip(tokens, tokens[1:]))
    pair_counts = Counter(pairs)
    return pair_counts.most_common(1)[0][0]

In [ ]:
tokens = [1, 2, 3, 2, 3, 2, 3]

pair = get_frequent_pair(tokens)

print("Most frequent pair:", pair)

Most frequent pair: (2, 3)


In [ ]:
print(get_frequent_pair([10]))

None


## Checkpoint 2

Explain the following line:

```python
pairs = list(zip(tokens, tokens[1:]))
```
**This line generates all adjacent pairs of tokens in the tokens list**

Then explain why we use:

```python
Counter(pairs)
```
** It identifies the most frequent pair to merge next**


# Part 3: Replacing a Pair

In [ ]:
def replace_pair(tokens, pair, new_id):
    result = []
    i = 0

    while i < len(tokens):

        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            result.append(new_id)
            i += 2

        else:
            result.append(tokens[i])
            i += 1

    return result

In [ ]:
tokens = [1, 2, 3, 2, 3, 4]

new_tokens = replace_pair(
    tokens,
    (2, 3),
    256
)

print("Before:", tokens)
print("After :", new_tokens)

Before: [1, 2, 3, 2, 3, 4]
After : [1, 256, 256, 4]


## Checkpoint 3

Why do we use:

```python
i += 2
```

when a pair is found?

**The pair is replaced by a new id so the index needs to advance to check for the next pair**

What would happen if we used:

```python
i += 1
```

instead?

**The replace_pair function would only advance the index by one position. This means that the second token of the already merged pair would immediately become the first token of the next potential pair to be checked**


# Task 4: Write train_bpe()

In [ ]:
def train_bpe(tokens, num_merges):
    # Store the learned merge rules.
    merges = {}

    # Make a copy so we do not modify the caller's list.
    tokens = tokens.copy()

    # New token IDs start after the 256 possible byte values.
    for new_id in range(256, 256 + num_merges):

        # Find the most frequent neighboring pair.
        pair = get_frequent_pair(tokens)

        # Stop if no pair can be formed.
        if pair is None:
            break

        # Replace the pair with the new token ID.
        tokens = replace_pair(tokens, pair, new_id)

        # Remember the merge rule.
        merges[pair] = new_id

    return merges

In [ ]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(training_text)

print("Training tokens:")
print(training_tokens)

Training tokens:
[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]


In [ ]:
merges = train_bpe(
    training_tokens,
    num_merges=10
)

In [ ]:
print("Learned merges:")

for pair, new_id in merges.items():
    print(pair, "→", new_id)

Learned merges:
(116, 104) → 256
(256, 101) → 257
(257, 32) → 258
(97, 116) → 259
(258, 99) → 260
(260, 259) → 261
(261, 32) → 262
(262, 105) → 263
(263, 110) → 264
(264, 32) → 265


In [ ]:
def train_bpe_observable(tokens, num_merges):
    merges = {}
    tokens = tokens.copy()

    for new_id in range(256, 256 + num_merges):

        pair = get_frequent_pair(tokens)

        if pair is None:
            break

        tokens = replace_pair(tokens, pair, new_id)

        merges[pair] = new_id

        print("Merge:", pair, "→", new_id)
        print("Tokens:", tokens)
        print()

    return merges

In [ ]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(training_text)

merges = train_bpe_observable(
    training_tokens,
    num_merges=10
)

Merge: (116, 104) → 256
Tokens: [256, 101, 32, 99, 97, 116, 32, 105, 110, 32, 256, 101, 32, 104, 97, 116]

Merge: (256, 101) → 257
Tokens: [257, 32, 99, 97, 116, 32, 105, 110, 32, 257, 32, 104, 97, 116]

Merge: (257, 32) → 258
Tokens: [258, 99, 97, 116, 32, 105, 110, 32, 258, 104, 97, 116]

Merge: (97, 116) → 259
Tokens: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]

Merge: (258, 99) → 260
Tokens: [260, 259, 32, 105, 110, 32, 258, 104, 259]

Merge: (260, 259) → 261
Tokens: [261, 32, 105, 110, 32, 258, 104, 259]

Merge: (261, 32) → 262
Tokens: [262, 105, 110, 32, 258, 104, 259]

Merge: (262, 105) → 263
Tokens: [263, 110, 32, 258, 104, 259]

Merge: (263, 110) → 264
Tokens: [264, 32, 258, 104, 259]

Merge: (264, 32) → 265
Tokens: [265, 258, 104, 259]



## Checkpoint 4

Explain the difference between:

```python
get_frequent_pair(tokens)
```

and:

```python
replace_pair(tokens, pair, new_id)
```
**get_frequent_pair identifies what to change, while replace_pair executes that change**

Then explain what information is stored in:

```python
merges
```
**The merges dictionary stores the learned merge rules during the BPE training process**

# Task 5: Modify Training to Build the Vocabulary

In [ ]:
def train_bpe(tokens, num_merges):
    # Store the learned merge rules.
    merges = {}

    # Create the initial vocabulary.
    # IDs 0-255 represent individual byte values.
    vocab = {
        i: bytes([i])
        for i in range(256)
    }

    # Make a copy of the input tokens.
    tokens = tokens.copy()

    # Create new token IDs starting from 256.
    for new_id in range(256, 256 + num_merges):

        # Find the most frequent pair.
        pair = get_frequent_pair(tokens)

        # Stop if no pair exists.
        if pair is None:
            break

        # Merge the pair.
        tokens = replace_pair(tokens, pair, new_id)

        # Remember the merge rule.
        merges[pair] = new_id

        # Build the byte representation of the new token.
        vocab[new_id] = (
            vocab[pair[0]]
            + vocab[pair[1]]
        )

    return merges, vocab

In [ ]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(training_text)

merges, vocab = train_bpe(
    training_tokens,
    num_merges=10
)

In [ ]:
print("Number of merges:", len(merges))
print("Vocabulary size:", len(vocab))

Number of merges: 10
Vocabulary size: 266


In [ ]:
for token_id in sorted(vocab):
    if token_id >= 256:
        print(token_id, "→", vocab[token_id])

256 → b'th'
257 → b'the'
258 → b'the '
259 → b'at'
260 → b'the c'
261 → b'the cat'
262 → b'the cat '
263 → b'the cat i'
264 → b'the cat in'
265 → b'the cat in '


## Checkpoint 5

Why does the vocabulary contain:

```text
0–255
```

from the beginning?

**Because these directly correspond to all possible single-byte values in the UTF-8 encoding**

Why do new tokens start at:

```text
256
```
**to avoid conflicts with the existing base token IDs**

What does an entry such as:

```python
vocab[256]
```

represent?

**the byte sequence that corresponds to the token ID 256**

# Task 6: Write encode()

In [ ]:
def encode(text, merges):
    # Start with individual UTF-8 byte tokens.
    tokens = text_to_bytes(text)

    # Apply the learned merge rules in order.
    for pair, new_id in merges.items():

        # Replace occurrences of the current pair.
        tokens = replace_pair(
            tokens,
            pair,
            new_id
        )

    return tokens

In [ ]:
text = "the cat in the hat"

ids = encode(
    text,
    merges
)

print("Text:", text)
print("Token IDs:", ids)

Text: the cat in the hat
Token IDs: [265, 258, 104, 259]


In [ ]:
new_text = "the hat"

new_ids = encode(
    new_text,
    merges
)

print("Text:", new_text)
print("Token IDs:", new_ids)

Text: the hat
Token IDs: [258, 104, 259]


# Task 7: Write decode()

In [ ]:
def decode(token_ids, vocab):
    # Start with an empty byte string.
    result = b""

    # Look up every token.
    for token_id in token_ids:

        # Add its byte representation.
        result += vocab[token_id]

    # Convert the final bytes back into text.
    return result.decode("utf-8")

In [ ]:
ids = encode(
    "the cat in the hat",
    merges
)

decoded_text = decode(
    ids,
    vocab
)

print("Encoded:", ids)
print("Decoded:", decoded_text)

Encoded: [265, 258, 104, 259]
Decoded: the cat in the hat


In [ ]:
assert decoded_text == "the cat in the hat"

# Checkpoint 7

Explain why decoding does not need to manually perform:

```text
257
 ↓
256 + 101
 ↓
116 + 104 + 101
```

in our implementation.

**because of how our vocab dictionary is constructed and utilized. Whenever a new token ID is created by merging a pair, its corresponding byte representation is immediately built and stored in the vocab dictionary**

What does the vocabulary allow us to do instead?

**The vocabulary allows us to perform a direct lookup for any token ID**

# Part 8: Seeing the Complete Training Process

In [ ]:
def train_bpe_observable(tokens, num_merges):
    merges = {}

    vocab = {
        i: bytes([i])
        for i in range(256)
    }

    tokens = tokens.copy()

    for new_id in range(256, 256 + num_merges):

        pair = get_frequent_pair(tokens)

        if pair is None:
            break

        tokens = replace_pair(
            tokens,
            pair,
            new_id
        )

        merges[pair] = new_id

        vocab[new_id] = (
            vocab[pair[0]]
            + vocab[pair[1]]
        )

        print(
            f"Merge: {pair} → {new_id}"
        )

        print(
            f"Token represents: {vocab[new_id]!r}"
        )

        print(
            f"Current tokens: {tokens}"
        )

        print()

    return merges, vocab

In [ ]:
training_text = "the cat in the hat"

training_tokens = text_to_bytes(
    training_text
)

merges, vocab = train_bpe_observable(
    training_tokens,
    num_merges=10
)

Merge: (116, 104) → 256
Token represents: b'th'
Current tokens: [256, 101, 32, 99, 97, 116, 32, 105, 110, 32, 256, 101, 32, 104, 97, 116]

Merge: (256, 101) → 257
Token represents: b'the'
Current tokens: [257, 32, 99, 97, 116, 32, 105, 110, 32, 257, 32, 104, 97, 116]

Merge: (257, 32) → 258
Token represents: b'the '
Current tokens: [258, 99, 97, 116, 32, 105, 110, 32, 258, 104, 97, 116]

Merge: (97, 116) → 259
Token represents: b'at'
Current tokens: [258, 99, 259, 32, 105, 110, 32, 258, 104, 259]

Merge: (258, 99) → 260
Token represents: b'the c'
Current tokens: [260, 259, 32, 105, 110, 32, 258, 104, 259]

Merge: (260, 259) → 261
Token represents: b'the cat'
Current tokens: [261, 32, 105, 110, 32, 258, 104, 259]

Merge: (261, 32) → 262
Token represents: b'the cat '
Current tokens: [262, 105, 110, 32, 258, 104, 259]

Merge: (262, 105) → 263
Token represents: b'the cat i'
Current tokens: [263, 110, 32, 258, 104, 259]

Merge: (263, 110) → 264
Token represents: b'the cat in'
Current tokens

# Task 8: Run the Complete Pipeline

In [ ]:
text = "the cat in the hat"

In [ ]:
tokens = text_to_bytes(text)

print("Initial tokens:")
print(tokens)

Initial tokens:
[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]


In [ ]:
merges, vocab = train_bpe(
    tokens,
    num_merges=10
)

In [ ]:
ids = encode(
    text,
    merges
)

print("Token IDs:")
print(ids)

Token IDs:
[265, 258, 104, 259]


In [ ]:
decoded = decode(
    ids,
    vocab
)

print("Decoded text:")
print(decoded)

Decoded text:
the cat in the hat


In [ ]:
assert decoded == text

# Final Test

In [ ]:
print("Original :", text)
print("Token IDs:", ids)
print("Decoded  :", decoded)

Original : the cat in the hat
Token IDs: [265, 258, 104, 259]
Decoded  : the cat in the hat


# Part 10: Experiment with the Tokenizer

Experiment 2: Change the training text

Experiment 3: Encode new text

# Part 12: Complete Script

In [ ]:
from collections import Counter


# --------------------------------------------------
# 1. Convert text to UTF-8 byte IDs
# --------------------------------------------------

def text_to_bytes(text):
    """
    Convert a Python string into a list of UTF-8 byte IDs.

    Each byte has a value from 0 to 255.
    """
    byte_array = bytearray(text, "utf-8")

    return list(byte_array)


# --------------------------------------------------
# 2. Find the most frequent neighboring pair
# --------------------------------------------------

def get_frequent_pair(tokens):
    """
    Find and return the most frequent neighboring pair.

    Example:

        [1, 2, 3, 2, 3]

    produces:

        (1, 2)
        (2, 3)
        (3, 2)
        (2, 3)

    The most frequent pair is:

        (2, 3)
    """

    if len(tokens) < 2:
        return None

    pairs = list(zip(tokens, tokens[1:]))

    pair_counts = Counter(pairs)

    return pair_counts.most_common(1)[0][0]


# --------------------------------------------------
# 3. Replace a pair with a new token ID
# --------------------------------------------------

def replace_pair(tokens, pair, new_id):
    """
    Replace every occurrence of 'pair' with 'new_id'.

    Example:

        tokens = [1, 2, 3, 2, 3, 4]
        pair = (2, 3)
        new_id = 256

    result:

        [1, 256, 256, 4]
    """

    result = []

    i = 0

    while i < len(tokens):

        if (
            i < len(tokens) - 1
            and (tokens[i], tokens[i + 1]) == pair
        ):
            result.append(new_id)

            # We consumed two tokens.
            i += 2

        else:
            result.append(tokens[i])

            # We consumed one token.
            i += 1

    return result


# --------------------------------------------------
# 4. Train the BPE tokenizer
# --------------------------------------------------

def train_bpe(tokens, num_merges):
    """
    Learn BPE merge rules from training tokens.

    Returns:

        merges
        vocab
    """

    merges = {}

    # Initial vocabulary contains all 256 byte values.
    vocab = {
        i: bytes([i])
        for i in range(256)
    }

    # Work on a copy of the training tokens.
    tokens = tokens.copy()

    # New token IDs start at 256.
    for new_id in range(256, 256 + num_merges):

        # Find the most frequent pair.
        pair = get_frequent_pair(tokens)

        # Stop if there are no pairs.
        if pair is None:
            break

        # Replace the pair with the new token ID.
        tokens = replace_pair(
            tokens,
            pair,
            new_id
        )

        # Remember the merge rule.
        merges[pair] = new_id

        # Build the byte representation of the new token.
        vocab[new_id] = (
            vocab[pair[0]]
            + vocab[pair[1]]
        )

    return merges, vocab


# --------------------------------------------------
# 5. Encode text
# --------------------------------------------------

def encode(text, merges):
    """
    Convert text into token IDs using learned merges.
    """

    # Start with byte IDs.
    tokens = text_to_bytes(text)

    # Apply learned merges in order.
    for pair, new_id in merges.items():

        tokens = replace_pair(
            tokens,
            pair,
            new_id
        )

    return tokens


# --------------------------------------------------
# 6. Decode token IDs
# --------------------------------------------------

def decode(token_ids, vocab):
    """
    Convert token IDs back into a Python string.
    """

    result = b""

    for token_id in token_ids:

        result += vocab[token_id]

    return result.decode("utf-8")


# ==================================================
# EXECUTE THE COMPLETE PIPELINE
# ==================================================

# Training data
training_text = "the cat in the hat"

# Step 1: Convert training text to byte tokens
training_tokens = text_to_bytes(training_text)

print("Training tokens:")
print(training_tokens)
print()


# Step 2: Train BPE
merges, vocab = train_bpe(
    training_tokens,
    num_merges=10
)

print("Learned merges:")
for pair, new_id in merges.items():
    print(pair, "→", new_id)

print()


# Step 3: Encode text
ids = encode(
    training_text,
    merges
)

print("Encoded IDs:")
print(ids)
print()


# Step 4: Decode text
decoded = decode(
    ids,
    vocab
)

print("Decoded text:")
print(decoded)
print()


# Step 5: Verify correctness
assert decoded == training_text

print("Encoding and decoding successful!")

Training tokens:
[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]

Learned merges:
(116, 104) → 256
(256, 101) → 257
(257, 32) → 258
(97, 116) → 259
(258, 99) → 260
(260, 259) → 261
(261, 32) → 262
(262, 105) → 263
(263, 110) → 264
(264, 32) → 265

Encoded IDs:
[265, 258, 104, 259]

Decoded text:
the cat in the hat

Encoding and decoding successful!


# Part 13: Compare with tiktoken

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = "This is some text"

ids = tokenizer.encode(text)

print(ids)

[1212, 318, 617, 2420]


In [ ]:
decoded = tokenizer.decode(ids)

print(decoded)

This is some text


# Final Questions

Answer these questions after completing the lab.

### Question 1

Why do we start with 256 possible byte token IDs?

because these represent all the fundamental units that make up any UTF-8 encoded text

### Question 2

Why does the first newly created token have ID 256?

to avoid clashes with the existing 0-255 byte token IDs

### Question 3

What does the following merge mean?

```text
(116, 104) → 256
```
This merge means that the byte pair (116, 104) has been identified as a frequent sequence and is now represented by a new single token ID

### Question 4

What is the difference between BPE training and encoding?

BPE Training is the process of learning the merge rules and building the vocabulary while Encoding is the process of converting new input text into a sequence of token IDs using the merge rules and vocabulary learned during training

### Question 5

Why do we need to store the learned merges?

because the encode() function relies on these rules to convert new text into token IDs

### Question 6

Why do we need a vocabulary?

to be able to decode token IDs back into human-readable text

### Question 7

What does `decode()` do?

It iterates through the list of token IDs, looking up the byte representation for each ID in the vocabulary. It then concatenates these byte representations to form a single byte string, which is decoded into a UTF-8 string

### Question 8

What happens when we increase `num_merges`?

the BPE algorithm will perform more merging operations

### Question 9

Why can the tokenizer trained on:

```text
"the cat in the hat"
```

also encode:

```text
"the hat"
```

?

because the individual bytes (b't', b'h', b'e', b' ', b'a') and the merged tokens derived from them (like b'th', b'the', b'at') are all present in the learned merges and vocab. The encode function applies these rules, regardless of whether the exact phrase was seen during training.

### Question 10

What are some differences between this educational tokenizer and a production tokenizer such as `tiktoken`?

The educational tokenizer strictly merges byte pairs. Production tokenizers like tiktoken often use more sophisticated subword segmentation algorithms that can handle diverse languages and punctuation in more nuanced ways.
Production tokenizers usually include complex preprocessing steps to prepare text before tokenization. The educational tokenizer only converts text to UTF-8 bytes.
